# A03 CPI SQL 与引用预检 · 2026-09-04
## tl;dr
完整18列只读转换查询、聚合、主键及空值预检通过。正式模型未修改，CPI最终验收仍FAIL。
查询：7680行、7616非空、64空值；MAX=47,954.24，AVG=375.80；主键重复=0。


## Context & Methods
粒度为iso3+month_end，40国，2010-01-31至2025-12-31。本次读取日期为2026-09-04（Asia/Shanghai）。
平台：Smartbi“可导入数据库”，input.v50_country_monthly_risk；正式模型MDL_XH202612_V50_COUNTRY_RESERVE。
### Key Assumptions
不加筛选、连接、去重或填零；仅cpi_index在聚合前CAST为DECIMAL(30,12)。
聚合只按界面两位小数和既有独立源文件基准比对，不宣称全精度相等。
SQL通过平台原生查询界面人工/浏览器执行；本笔记本只核验已保存证据，不重新查询数据库。
原表“编辑”菜单禁用，原因未确定；不据此断定账号权限不足。


## Data
输入为同目录SQL、原生DOM、复制SQL、引用JSON与前序源端基准；详见README.md。
当前模型引用取9/4受控导出；页面字段引用取8/31历史XML，不能充当最新完整租户依赖扫描。
执行环境：Python标准库；无nbformat/nbclient。本文件代码单元已在同一Python命名空间顺序执行并保存输出，
未在Jupyter内核验证。如需内核复验，安装jupyter/nbconvert后在本目录运行：
`python -m jupyter nbconvert --execute --to notebook --inplace CPI_SQL_PREFLIGHT.ipynb`。


In [1]:
from pathlib import Path
import json
import runpy
folder = Path.cwd()
assert (folder / 'verify_preflight.py').is_file(), '请以本证据目录为工作目录运行'


### 1. 查看平台实际执行的查询

In [2]:
for name in ['CPI_FULL_QUERY_CHECK.sql', '04_FULL_18_COLUMN_EXECUTED.sql', 'CPI_KEY_NULL_CHECK.sql']:
    print(name)
    print((folder / name).read_text(encoding='utf-8'))


CPI_FULL_QUERY_CHECK.sql
SELECT COUNT(*) AS total_rows,
       COUNT(cpi_index) AS non_null_rows,
       SUM(CASE WHEN cpi_index IS NULL THEN 1 ELSE 0 END) AS null_rows,
       SUM(cpi_index) AS numeric_sum,
       AVG(cpi_index) AS numeric_avg,
       MAX(cpi_index) AS numeric_max
FROM (
    SELECT iso3,
           month_end,
           fx_avg_lcu_per_usd,
           CAST(NULLIF(TRIM(cpi_index), '') AS DECIMAL(30,12)) AS cpi_index,
           fx_reserves_usd,
           reserve_import_months,
           imports_usd,
           source_id,
           source_frequency,
           fetch_date,
           data_version,
           is_proxy,
           is_imputed,
           run_id,
           fx_eom_lcu_per_usd,
           fx_eom_source,
           fx_avg_source,
           cpi_source
    FROM input.v50_country_monthly_risk
) cpi_cast_check

04_FULL_18_COLUMN_EXECUTED.sql
SELECT iso3,
       month_end,
       fx_avg_lcu_per_usd,
       CAST(NULLIF(TRIM(cpi_index), '') AS DECIMAL(30,12)) AS c

## Results
### 2. 核验原生结果、SQL一致性和保护文件哈希

In [3]:
checks = runpy.run_path(str(folder / 'verify_preflight.py'))
report = checks['verify']()
print(json.dumps({k: report[k] for k in ['sqlPreflight', 'formalRepairStatus', 'metrics', 'grainAndNullCheck', 'queryDiscardedWithoutSaving', 'formalModelPublished']}, ensure_ascii=False, indent=2))


{
  "sqlPreflight": "PASS",
  "formalRepairStatus": "FAIL_MAX_PENDING_REPAIR",
  "metrics": {
    "total_rows": "7,680",
    "non_null_rows": "7,616",
    "null_rows": "64",
    "numeric_sum": "2,862,057.90",
    "numeric_avg": "375.80",
    "numeric_max": "47,954.24"
  },
  "grainAndNullCheck": {
    "total_rows": "7,680",
    "country_count": "40",
    "first_month": "2010-01-31",
    "last_month": "2025-12-31",
    "null_key_rows": "0",
    "raw_null_rows": "64",
    "blank_string_rows": "0",
    "cast_null_rows": "64",
    "duplicate_key_groups": "0",
    "extra_duplicate_rows": "0"
  },
  "queryDiscardedWithoutSaving": true,
  "formalModelPublished": false
}


### 3. 核对依赖边界

In [4]:
refs = json.loads((folder / 'REFERENCE_AUDIT.json').read_text(encoding='utf-8'))
print('原字段数:', refs['sourceFieldCount'])
print('来源关系数:', len(refs['relationsInvolvingSource']))
print('历史直接CPI组件:', [(p['name'], c['id']) for p in refs['pages'] for c in p['portlets']])
print(refs['scope'])


原字段数: 18
来源关系数: 2
历史直接CPI组件: [('DB02_XH202612_V50_COUNTRY_DRILL', '50069196dac827c6732e076784c7fd80')]
Current 2026-09-04 model export and historical controlled six-page export; not a complete live tenant reference scan


## Takeaways
预检材料可共享，但不代表正式修复验收。主模型没有发布，测试SQL已不保存关闭。
下一步需确认保留18字段ID/2关系及DB02私有度量引用的原位数值化或受控替换入口。
正式上线后才复测原生MAX、AVG、趋势/筛选并交B独立复核。Excel状态、B签署和AI六题跳过状态不变。
